# SSL400 Sinhala Sign Language — MoViNet-A2 Two-Phase Training
## Phase 1 (Frozen Backbone) + Phase 2 (Backbone Fine-Tuning)

### Why MoViNet-A2 instead of I3D?
| Feature | I3D (Old) | MoViNet-A2 (New) |
|---|---|---|
| TF Format | TF1 (Legacy) | TF2 (Native) |
| Backbone Fine-Tuning | ❌ BLOCKED (ValueError) | ✅ FULLY SUPPORTED |
| Pre-trained on | Kinetics-400 | Kinetics-600 (richer!) |
| Phase 2 Expected Boost | N/A | +10% to +20% accuracy |

### Experiment Control
Change only `EXP_ID` in Cell 2 to switch between experiments:
- `EXP_ID = 1` → Baseline (No Enhancement)
- `EXP_ID = 2` → CLAHE + Gamma Correction
- `EXP_ID = 3` → Bilateral Filter
- `EXP_ID = 4` → Unsharp Masking
- `EXP_ID = 5` → Hybrid (Bilateral + CLAHE + Unsharp)

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Setup — ONLY CHANGE EXP_ID HERE ───────────────────────────────────
import os
os.chdir('/content/drive/MyDrive/ssl400_research_project')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import yaml
EXP_ID     = 1    # ← CHANGE THIS: 1, 2, 3, 4, or 5
BATCH_SIZE = 16   # Safe for Colab Pro A100 High-RAM (167GB)

with open('config.yaml') as f:
    config = yaml.safe_load(f)

print(f"Experiment {EXP_ID}: {config['experiments'][EXP_ID]['name']}")
print(f"Batch Size  : {BATCH_SIZE}")

In [ ]:
# ── Cell 3: Imports & GPU Verification ────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

import numpy as np
import random
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU Devices Available: {gpus}")
assert len(gpus) > 0, "ERROR: No GPU found! Go to Runtime -> Change runtime type -> A100 GPU"

seed = config['project']['seed']
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
print("Seeds set for reproducibility.")

In [ ]:
# ── Cell 4: Build Datasets ────────────────────────────────────────────────────
from src.data.tf_dataset_builder import build_dataset

num_classes   = config['model']['num_classes']
target_frames = config['video']['target_frames']
model_dir     = config['experiments'][EXP_ID]['model_dir']
log_dir       = config['experiments'][EXP_ID]['log_dir']

os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir,   exist_ok=True)

print(f'Building datasets for EXP{EXP_ID}: {config["experiments"][EXP_ID]["name"]}...')
train_ds = build_dataset('data/splits/train_split.csv', EXP_ID, BATCH_SIZE,
                          num_classes, target_frames, augment=True,  shuffle=True)
val_ds   = build_dataset('data/splits/val_split.csv',   EXP_ID, BATCH_SIZE,
                          num_classes, target_frames, augment=False, shuffle=False)
test_ds  = build_dataset('data/splits/test_split.csv',  EXP_ID, BATCH_SIZE,
                          num_classes, target_frames, augment=False, shuffle=False)
print('Datasets ready!')

In [ ]:
# ── Cell 5: Mixup Augmentation Helper ─────────────────────────────────────────
# Mixup blends two training samples together, forcing the model to learn
# smooth, interpolated decision boundaries instead of memorizing exact samples.
# Proven to reduce overfitting by ~2-5% on low-resource datasets like SSL400.

def mixup_data(x, y, alpha=0.2):
    """Apply Mixup augmentation to a batch of (video_clip, label) pairs."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = tf.shape(x)[0]
    index = tf.random.shuffle(tf.range(batch_size))
    mixed_x = lam * x + (1 - lam) * tf.gather(x, index)
    mixed_y = lam * y + (1 - lam) * tf.gather(y, index)
    return mixed_x, mixed_y

def apply_mixup_to_dataset(dataset, alpha=0.2):
    """Wrap a tf.data.Dataset to apply Mixup on each batch."""
    def mixup_map(x, y):
        mixed_x, mixed_y = tf.py_function(
            func=lambda bx, by: mixup_data(bx.numpy(), by.numpy(), alpha),
            inp=[x, y],
            Tout=[tf.float32, tf.float32]
        )
        mixed_x.set_shape(x.shape)
        mixed_y.set_shape(y.shape)
        return mixed_x, mixed_y
    return dataset.map(mixup_map, num_parallel_calls=tf.data.AUTOTUNE)

# Apply Mixup only to training data
train_ds_mixup = apply_mixup_to_dataset(train_ds, alpha=0.2)
print("Mixup augmentation applied to training dataset.")

In [ ]:
# ── Cell 6: PHASE 1 — Warm-Up Training (MoViNet Backbone Frozen) ──────────────
# Phase 1 trains ONLY the new 383-class classification head.
# The MoViNet-A2 backbone is frozen — preserving Kinetics-600 knowledge.
# This is fast and safe: can't destroy the pre-trained weights.

from src.models.movinet_builder import build_and_compile_phase1

PHASE1_EPOCHS   = config['training']['max_epochs_phase1']  # 50
PHASE1_LR       = config['training']['lr_phase1']           # 0.001
PHASE1_PATIENCE = 10

phase1_model_path = f"{model_dir}/best_model_phase1.keras"

print('Building MoViNet-A2 model (Frozen Backbone)...')
model = build_and_compile_phase1()

# --- Auto-Resume: loads best checkpoint if a previous session was interrupted ---
if os.path.exists(phase1_model_path):
    print(f"\nFOUND EXISTING PHASE 1 MODEL! Resuming from {phase1_model_path}")
    model.load_weights(phase1_model_path)
else:
    print("Starting Phase 1 from scratch.")

# Cosine Annealing LR: smoothly decays LR to near-zero over all epochs.
# Consistently outperforms ReduceLROnPlateau in research benchmarks.
cosine_schedule_p1 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=PHASE1_LR,
    decay_steps=PHASE1_EPOCHS * 186,  # steps per epoch approx (3000 train / batch_size)
    alpha=1e-6
)
model.optimizer.learning_rate = cosine_schedule_p1

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=PHASE1_PATIENCE,
        restore_best_weights=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=phase1_model_path,
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        f"{log_dir}/training_log_phase1.csv", append=True
    ),
]

print(f'Starting Phase 1 training ({PHASE1_EPOCHS} epochs, Cosine Annealing LR)...')
history_p1 = model.fit(
    train_ds_mixup,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks_p1,
    verbose=1,
)
print(f"Phase 1 complete! Best val_accuracy: {max(history_p1.history['val_accuracy']):.4f}")

In [ ]:
# ── Cell 7: PHASE 2 — Fine-Tuning (Unfreeze MoViNet Backbone) ────────────────
# THIS IS THE BIG UPGRADE OVER I3D!
# MoViNet is TF2 native — backbone fine-tuning is FULLY SUPPORTED.
# We unfreeze the entire backbone and fine-tune at a very small LR (1e-5).
# This adapts the Kinetics-600 motion features to Sinhala hand sign patterns.
# Expected accuracy boost: +10% to +20% over Phase 1 alone!

from src.models.movinet_builder import unfreeze_for_phase2

PHASE2_EPOCHS = config['training']['max_epochs_phase2']  # 30
PHASE2_LR     = config['training']['lr_phase2']           # 1e-5

phase2_model_path = f"{model_dir}/best_model_phase2.keras"

print("Unfreezing MoViNet-A2 backbone for Phase 2 fine-tuning...")
model, trainable = unfreeze_for_phase2(model, new_learning_rate=PHASE2_LR)
print(f"Trainable parameters after unfreeze: {trainable:,}")

# Cosine Annealing for Phase 2 — even gentler decay to protect pre-trained weights
cosine_schedule_p2 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=PHASE2_LR,
    decay_steps=PHASE2_EPOCHS * 186,
    alpha=1e-7
)
model.optimizer.learning_rate = cosine_schedule_p2

# Check for existing Phase 2 model (for auto-resume)
if os.path.exists(phase2_model_path):
    print(f"FOUND EXISTING PHASE 2 MODEL! Resuming from {phase2_model_path}")
    model.load_weights(phase2_model_path)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=phase2_model_path,
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        f"{log_dir}/training_log_phase2.csv", append=True
    ),
]

print(f'Starting Phase 2 fine-tuning ({PHASE2_EPOCHS} epochs at LR={PHASE2_LR})...')
history_p2 = model.fit(
    train_ds,         # No Mixup in Phase 2 — clean gradients for careful fine-tuning
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks_p2,
    verbose=1,
)
print(f"Phase 2 complete! Best val_accuracy: {max(history_p2.history['val_accuracy']):.4f}")

In [ ]:
# ── Cell 8: Save Final Model & Test Set Evaluation ────────────────────────────
final_model_path = f'{model_dir}/best_model.keras'
model.save(final_model_path)
print(f'Final model saved to {final_model_path}')

print('\nEvaluating on HELD-OUT TEST SET...')
test_results = model.evaluate(test_ds, verbose=1)
print(f'\nFINAL TEST RESULTS for EXP{EXP_ID}: {config["experiments"][EXP_ID]["name"]}')
print(f'  Test Loss          : {test_results[0]:.4f}')
print(f'  Test Top-1 Accuracy: {test_results[1]*100:.2f}%')
print(f'  Test Top-5 Accuracy: {test_results[2]*100:.2f}%')

# Compare against original SOTA
sota = 88.23
diff = test_results[1]*100 - sota
print(f'\n  vs. Original SOTA (88.23%): {diff:+.2f}%')
if diff > 0:
    print(f'  ✅ BEAT SOTA BY {diff:.2f}%!')
else:
    print(f'  ❌ Below SOTA by {abs(diff):.2f}% — try Phase 2 longer or increase patience')

# Save test results to text file for research report
with open(f'{log_dir}/final_test_results.txt', 'w') as f:
    f.write(f'EXP{EXP_ID}: {config["experiments"][EXP_ID]["name"]}\n')
    f.write(f'Model       : MoViNet-A2 (Kinetics-600) + Two-Phase Training\n')
    f.write(f'Test Loss   : {test_results[0]:.4f}\n')
    f.write(f'Top-1 Acc   : {test_results[1]*100:.2f}%\n')
    f.write(f'Top-5 Acc   : {test_results[2]*100:.2f}%\n')
    f.write(f'vs SOTA     : {diff:+.2f}%\n')
print(f'Results saved to {log_dir}/final_test_results.txt')

In [ ]:
# ── Cell 9: Training Curves Plot (Both Phases) ────────────────────────────────
import matplotlib.pyplot as plt

# Combine Phase 1 + Phase 2 history
combined_acc      = history_p1.history['accuracy']      + history_p2.history['accuracy']
combined_val_acc  = history_p1.history['val_accuracy']  + history_p2.history['val_accuracy']
combined_loss     = history_p1.history['loss']          + history_p2.history['loss']
combined_val_loss = history_p1.history['val_loss']      + history_p2.history['val_loss']
total_epochs      = range(1, len(combined_acc) + 1)
phase2_start      = len(history_p1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'EXP{EXP_ID}: {config["experiments"][EXP_ID]["name"]} — MoViNet-A2 Two-Phase Training',
    fontsize=14
)

# Accuracy plot
axes[0].plot(total_epochs, combined_acc,     'b-',  label='Train Accuracy')
axes[0].plot(total_epochs, combined_val_acc, 'r-',  label='Val Accuracy')
axes[0].axvline(x=phase2_start, color='green', linestyle='--', label='Phase 2 Start (Unfreeze)')
axes[0].axhline(y=0.8823, color='orange', linestyle=':', label='Previous SOTA (88.23%)')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(total_epochs, combined_loss,     'b-', label='Train Loss')
axes[1].plot(total_epochs, combined_val_loss, 'r-', label='Val Loss')
axes[1].axvline(x=phase2_start, color='green', linestyle='--', label='Phase 2 Start (Unfreeze)')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

os.makedirs('results/figures', exist_ok=True)
plt.tight_layout()
fig.savefig(f'results/figures/exp{EXP_ID}_two_phase_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')